In [120]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [121]:
from data_curation import DataLoader, SafeGroupKFold
from i3l_statistics import Statistics
from i3l_ml import ML
from enums import *
import json
import pandas as pd
from sklearn.model_selection import GroupKFold

# Load data and separate in train, test and external set

In [122]:
dl = DataLoader()
ml = ML()
stats = Statistics()

In [124]:
modes = [
    Mode.RWD, 
    # Mode.DP, 
    # Mode.FMRAD, 
    # Mode.PYRAD, 
    # Mode.GEN
]

dataset = dl.create_dataset(
    modes=modes, 
    outcome='OS MONTHS',
    cohort=23,
    subanalysis=Subanalysis.CLASSIC
)

In [125]:
with open('split.json', 'r') as f:
    split = json.load(f)

train_set = dataset[dataset['Subject'].isin(split['TRAIN_SET'])].set_index('Subject')
test_set = dataset[dataset['Subject'].isin(split['TEST_SET'])].set_index('Subject')
ext_set = dataset[dataset['Subject'].str.startswith('UOC')].set_index('Subject')

In [126]:
X_train, y_train = train_set.drop(columns=['OS MONTHS']), train_set['OS MONTHS']
X_test, y_test = test_set.drop(columns=['OS MONTHS']), test_set['OS MONTHS']
X_ext, y_ext = ext_set.drop(columns=['OS MONTHS']), ext_set['OS MONTHS']

In [127]:
outcome_name = Outcome.OS_6

y_train = dl.get_outcome(
    outcome=y_train,
    outcome_name=outcome_name,
)
y_test = dl.get_outcome(
    outcome=y_test,
    outcome_name=outcome_name,
)
y_ext = dl.get_outcome(
    outcome=y_ext,
    outcome_name=outcome_name,
)

In [128]:
with open('submodel_features.json', 'r') as f:
    submodel_features = json.load(f)
    submodel_features = [f for f in submodel_features if f in X_train.columns]

X_train = X_train.drop(columns=submodel_features)
X_test = X_test.drop(columns=submodel_features)
X_ext = X_ext.drop(columns=submodel_features)

In [129]:
X_train_imputed, imputer = dl.impute_df(X_train)
X_test_imputed, imputer = dl.impute_df(X_test, imputer=imputer)
X_ext_imputed, imputer = dl.impute_df(X_ext, imputer=imputer)

In [130]:
X_train_scaled, scaler, to_standard_normalize, to_log_normalize = dl.normalize(X_train_imputed)
X_test_scaled, _, _, _ = dl.normalize(X_test_imputed, scaler=scaler, to_standard_normalize=to_standard_normalize, to_log_normalize=to_log_normalize)
X_ext_scaled, _, _, _ = dl.normalize(X_ext_imputed, scaler=scaler, to_standard_normalize=to_standard_normalize, to_log_normalize=to_log_normalize)

6 features log normalized
11 features standardized
6 features log normalized
11 features standardized
6 features log normalized
11 features standardized


# Train and evaluate

In [131]:
train_folds = dl.get_loco_folds(pd.Series(train_set.index))

cv = SafeGroupKFold(n_splits=len(train_folds.unique()))

def get_split():
    return cv.split(X_train_scaled, y_train, groups=train_folds)

In [ ]:
model = ml.train_classification_model(
    X=X_train_scaled, 
    y=y_train,
    model_name=Model.RF,
    cv=get_split,
    select_features=True
)
selected_features = model.feature_names_in_

c:\Users\aferr\miniconda3\envs\ml\Lib\site-packages\skopt\space\space.py:116: UserWarning: Dimension [100, 150] was inferred to Integer(low=100, high=150, prior='uniform', transform='identity'). In upcoming versions of scikit-optimize, it will be inferred to Categorical(categories=(100, 150), prior=None). See the documentation of the check_dimension function for the upcoming API.
  warnings.warn(
c:\Users\aferr\miniconda3\envs\ml\Lib\site-packages\skopt\space\space.py:116: UserWarning: Dimension [10, 15] was inferred to Integer(low=10, high=15, prior='uniform', transform='identity'). In upcoming versions of scikit-optimize, it will be inferred to Categorical(categories=(10, 15), prior=None). See the documentation of the check_dimension function for the upcoming API.
  warnings.warn(
c:\Users\aferr\Desktop\Albi\Code\i3lung\ml_lung\for_publication\mlef_pipeline\data_curation.py:203: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer k

In [133]:
from sklearn.metrics import roc_auc_score

In [134]:
X_train_scaled = X_train_scaled[selected_features]
X_test_scaled = X_test_scaled[selected_features]
X_ext_scaled = X_ext_scaled[selected_features]

In [135]:
y_pred_train = model.predict_proba(X_train_scaled)[:, 1]
y_pred_test = model.predict_proba(X_test_scaled)[:, 1]
y_pred_ext = model.predict_proba(X_ext_scaled)[:, 1]

train_auc = stats.get_auc_ci(
    y_true=y_train, 
    y_pred=y_pred_train,
    alpha=0.95
)
test_auc = stats.get_auc_ci(
    y_true=y_test, 
    y_pred=y_pred_test,
    alpha=0.95
)
ext_auc = stats.get_auc_ci(
    y_true=y_ext, 
    y_pred=y_pred_ext,
    alpha=0.95
)

print(f"Train AUC: {train_auc}")
print(f"Test AUC: {test_auc}")
print(f"External AUC: {ext_auc}")

Train AUC: 0.774 ± 0.024
Test AUC: 0.630 ± 0.070
External AUC: 0.604 ± 0.077
